# Component 3 — Cross-Year Semantic Harmonization

This notebook converts the separate staged fisheries artifacts into one canonical schema using evidence-based assumptions. It preserves every staged record in an all-records artifact and creates a stricter modelling-candidate view. It does not calculate CPUE or join environmental data.

## Adopted assumptions

1. `kg` and `weight` represent catch weight in kilograms.
2. `qty` and `amount` represent a count-like quantity, not weight.
3. A row represents a species catch line within a fishing event or trip.
4. Fishing/set date fields are preferred over departure or arrival dates.
5. 2017 `longi` contains latitude-like values and `latti` contains longitude-like values.
6. 2020 latitude/longitude labels are swapped when their observed numeric ranges support that interpretation.
7. The 2022 issue sheet remains available in the all-records artifact but is excluded from modelling candidates.
8. Exact duplicates across canonical analytical fields are excluded only from the modelling-candidate view.
9. Comparable effort is unavailable across years, so the provisional target is catch weight rather than CPUE.

In [ ]:
import importlib.util, subprocess, sys
packages = {'pandas':'pandas>=2.0,<3','numpy':'numpy>=1.24,<3','pyarrow':'pyarrow>=14,<22','matplotlib':'matplotlib>=3.7,<4'}
missing = [spec for module, spec in packages.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
print('Harmonization dependencies are available.')


In [ ]:
from __future__ import annotations
import gc, json, os, re, sys
from pathlib import Path
import matplotlib
if 'IPython' not in sys.modules:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
pd.set_option('display.max_columns', 60)


## 1. Configure Colab or local paths

In [ ]:
def find_component_root(start: Path) -> Path:
    for anchor in [start.resolve(), *start.resolve().parents]:
        for candidate in [anchor, anchor/'component - 3', anchor/'J26-DS-350'/'component - 3']:
            if (candidate/'Notebooks').is_dir() and (candidate/'src').is_dir(): return candidate.resolve()
    raise FileNotFoundError('Component 3 repository root was not found.')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/FishGuard-C3')
    INTERIM_DIR = ROOT/'data'/'interim'
    PROCESSED_DIR = ROOT/'data'/'processed'
    EVIDENCE_DIR = ROOT/'outputs'/'audit'/'harmonization'
    FIGURE_DIR = ROOT/'outputs'/'figures'/'harmonization'
else:
    COMPONENT_ROOT = find_component_root(Path.cwd())
    INTERIM_DIR = Path(os.getenv('FISHGUARD_C3_INTERIM_INPUT', str(COMPONENT_ROOT/'data'/'interim'))).resolve()
    PROCESSED_DIR = Path(os.getenv('FISHGUARD_C3_PROCESSED_OUTPUT', str(COMPONENT_ROOT/'data'/'processed'))).resolve()
    EVIDENCE_DIR = Path(os.getenv('FISHGUARD_C3_HARMONIZATION_OUTPUT', str(COMPONENT_ROOT/'outputs'/'audit'/'harmonization'))).resolve()
    FIGURE_DIR = Path(os.getenv('FISHGUARD_C3_HARMONIZATION_FIGURES', str(COMPONENT_ROOT/'outputs'/'figures'/'harmonization'))).resolve()
for directory in [PROCESSED_DIR,EVIDENCE_DIR,FIGURE_DIR]: directory.mkdir(parents=True,exist_ok=True)
print('Runtime:', 'Google Colab' if IN_COLAB else 'Local Windows')
print('Interim input:', INTERIM_DIR)


## 2. Year-specific semantic mapping

In [ ]:
MAPPINGS = [
 {'year':2017,'file':'fisheries_2017_staged.parquet','species':'catch','weight':'kg','quantity':'qty','vessel':'reg_num','gear':'gear','date_mode':'ymd','month':'unnamed_2','day':'unnamed_3','lat':'longi','lon':'latti'},
 {'year':2018,'file':'fisheries_2018_staged.parquet','species':'fish_name','weight':'kg','quantity':'amount','vessel':'boat_no','gear':'catch_type','date_mode':'ymd_fields','year_field':'y','month':'m','day':'d','lat':'lat','lon':'long'},
 {'year':2019,'file':'fisheries_2019_staged.parquet','species':'fish_name','weight':'kg','quantity':'amount','vessel':'boat_no','gear':'catch_type','date_mode':'ymd','month':'month','day':'date_of_set','lat':'latitude','lon':'longetude'},
 {'year':2020,'file':'fisheries_2020_staged.parquet','species':'name','weight':'weight','quantity':'qty','vessel':'vessel_id','gear':'geartype','date_mode':'field','date':'dateandtime','lat':'longitude','lon':'latttitude','direction':'gps_derection'},
 {'year':2021,'file':'fisheries_2021_staged.parquet','species':'name','weight':'weight','quantity':'qty','vessel':'vessel_id','gear':'geartype','date_mode':'field','date':'date_time','lat':'start_n','lon':'start_e','direction':'startgpsderec'},
 {'year':2022,'file':'fisheries_2022_staged.parquet','species':'name','weight':'weight','quantity':'qty','vessel':'vessel_id','gear':'geartype','date_mode':'field','date':'date_time','lat':'lattude','lon':'longitude','direction':'lattude_direction'},
 {'year':2022,'file':'fisheries_2022_issues.parquet','species':'name','weight':'weight','quantity':'qty','vessel':'vessel_id','gear':'geartype','date_mode':'field','date':'date_time','lat':'lattitude','lon':'longitude','direction':'lattude_direction'},
 {'year':2023,'file':'fisheries_2023_staged.parquet','species':'name','weight':'weight','quantity':'qty','vessel':'vessel_id','gear':'geartype','date_mode':'field','date':'date_time','lat':'start_n','lon':'start_e'},
 {'year':2024,'file':'fisheries_2024_staged.parquet','species':'name','weight':'weight','quantity':'qty','vessel':'vessel_id','gear':'geartype','date_mode':'field','date':'date_time','lat':'start_n','lon':'start_e','direction':'startgpsderec'},
]
display(pd.DataFrame(MAPPINGS))


## 3. Parsing and harmonization functions

In [ ]:
def text(series): return series.astype('string').str.strip().str.replace(r'\s+',' ',regex=True)
def numeric(series): return pd.to_numeric(series,errors='coerce')

def coordinate(series):
    raw=text(series); direct=pd.to_numeric(raw,errors='coerce')
    pattern=raw.str.extract(r'(?i)^\s*([NSEW])?\s*(\d{1,3})(?:[^0-9]+(\d+(?:\.\d+)?))?')
    degrees=pd.to_numeric(pattern[1],errors='coerce'); minutes=pd.to_numeric(pattern[2],errors='coerce').fillna(0)
    parsed=degrees+minutes/60; hemisphere=pattern[0].str.upper()
    parsed=parsed.where(~hemisphere.isin(['S','W']),-parsed)
    return direct.fillna(parsed)

def apply_latitude_direction(latitude,direction):
    if direction is None: return latitude
    d=text(direction).str.lower(); absolute=latitude.abs()
    return latitude.where(~d.str.startswith('s',na=False),-absolute).where(~d.str.startswith('n',na=False),absolute)

def event_date(frame,mapping):
    if mapping['date_mode']=='field': return pd.to_datetime(frame[mapping['date']],errors='coerce',dayfirst=True,format='mixed')
    year=numeric(frame[mapping['year_field']]) if mapping['date_mode']=='ymd_fields' else pd.Series(mapping['year'],index=frame.index)
    return pd.to_datetime({'year':year,'month':numeric(frame[mapping['month']]),'day':numeric(frame[mapping['day']])},errors='coerce')

def harmonize(frame,mapping):
    out=pd.DataFrame(index=frame.index)
    for field in ['source_year_label','source_file','source_sheet','source_role','source_row_number']:
        out[field]=frame[field]
    out['species']=text(frame[mapping['species']]).str.lower()
    out['catch_weight_kg']=numeric(frame[mapping['weight']])
    out['quantity_count_candidate']=numeric(frame[mapping['quantity']])
    out['vessel_id']=text(frame[mapping['vessel']]).str.upper()
    out['gear']=text(frame[mapping['gear']]).str.lower()
    out['event_date']=event_date(frame,mapping)
    out['latitude']=coordinate(frame[mapping['lat']])
    out['longitude']=coordinate(frame[mapping['lon']])
    if mapping.get('direction'): out['latitude']=apply_latitude_direction(out['latitude'],frame[mapping['direction']])
    out['date_valid']=out['event_date'].between(pd.Timestamp('2017-01-01'),pd.Timestamp('2024-12-31'))
    out['coordinates_valid']=out['latitude'].between(-90,90)&out['longitude'].between(-180,180)
    out['weight_valid']=out['catch_weight_kg'].notna()&(out['catch_weight_kg']>=0)
    out['species_valid']=out['species'].notna()&out['species'].ne('')
    key=['source_year_label','vessel_id','event_date','latitude','longitude','gear']
    complete=out[key].notna().all(axis=1)
    out['event_id']=pd.Series(pd.NA,index=out.index,dtype='string')
    out.loc[complete,'event_id']='EVT-'+pd.util.hash_pandas_object(out.loc[complete,key],index=False).astype(str)
    return out


## 4. Harmonize and combine staged artifacts

In [ ]:
parts=[]; source_summary=[]
for mapping in MAPPINGS:
    path=INTERIM_DIR/mapping['file']
    if not path.is_file(): raise FileNotFoundError(f'Missing staged input: {path}. Run 02_data_cleaning.ipynb first.')
    staged=pd.read_parquet(path); canonical=harmonize(staged,mapping)
    parts.append(canonical)
    source_summary.append({'year':mapping['year'],'file':mapping['file'],'rows':len(canonical),
        'valid_dates':int(canonical.date_valid.sum()),'valid_coordinates':int(canonical.coordinates_valid.sum()),
        'valid_weights':int(canonical.weight_valid.sum()),'valid_species':int(canonical.species_valid.sum())})
    del staged,canonical; gc.collect()
all_records=pd.concat(parts,ignore_index=True); del parts; gc.collect()
duplicate_key=['source_year_label','vessel_id','event_date','latitude','longitude','gear','species','catch_weight_kg','quantity_count_candidate']
all_records['exact_canonical_duplicate']=all_records.duplicated(duplicate_key,keep='first')
all_records['core_fields_valid']=all_records[['date_valid','coordinates_valid','weight_valid','species_valid']].all(axis=1)
all_records['model_candidate']=all_records.core_fields_valid & all_records.source_role.eq('candidate_observations') & ~all_records.exact_canonical_duplicate
model_candidates=all_records.loc[all_records.model_candidate].copy()
source_summary=pd.DataFrame(source_summary)
display(source_summary); print('All records:',len(all_records)); print('Model candidates:',len(model_candidates))


## 5. Reconciliation and harmonization evidence

In [ ]:
assert len(all_records)==source_summary.rows.sum()
assert model_candidates.model_candidate.all()
summary=(all_records.groupby(['source_year_label','source_role'],dropna=False).agg(
    rows=('source_row_number','size'),valid_dates=('date_valid','sum'),valid_coordinates=('coordinates_valid','sum'),
    valid_weights=('weight_valid','sum'),valid_species=('species_valid','sum'),
    exact_duplicates=('exact_canonical_duplicate','sum'),model_candidates=('model_candidate','sum')).reset_index())
display(summary)
print('Reconciliation passed; the all-records artifact retains every staged row.')


## 6. Viva-ready harmonization plots

In [ ]:
plot=summary.groupby('source_year_label',as_index=False).sum(numeric_only=True)
fig,axes=plt.subplots(1,2,figsize=(15,5))
axes[0].bar(plot.source_year_label.astype(str),plot.rows,label='All staged rows',color='#4c78a8')
axes[0].bar(plot.source_year_label.astype(str),plot.model_candidates,label='Model candidates',color='#f58518')
axes[0].set_title('Record retention after semantic harmonization'); axes[0].set_ylabel('Rows'); axes[0].legend(); axes[0].grid(axis='y',alpha=.25)
validity=pd.DataFrame({'Date':100*plot.valid_dates/plot.rows,'Coordinates':100*plot.valid_coordinates/plot.rows,
    'Weight':100*plot.valid_weights/plot.rows,'Species':100*plot.valid_species/plot.rows},index=plot.source_year_label.astype(str))
validity.plot(kind='bar',ax=axes[1]); axes[1].set_title('Canonical-field validity by year'); axes[1].set_ylabel('Percent of rows'); axes[1].set_ylim(0,105); axes[1].grid(axis='y',alpha=.25)
fig.tight_layout(); path=FIGURE_DIR/'harmonization_summary.png'; fig.savefig(path,dpi=200,bbox_inches='tight'); plt.show()
sample=model_candidates.dropna(subset=['longitude','latitude']).sample(min(50000,len(model_candidates)),random_state=42)
fig,ax=plt.subplots(figsize=(8,7)); ax.scatter(sample.longitude,sample.latitude,s=3,alpha=.2)
ax.set_title('Spatial coverage of harmonized modelling candidates'); ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); ax.grid(alpha=.25)
path2=FIGURE_DIR/'harmonized_spatial_coverage.png'; fig.savefig(path2,dpi=200,bbox_inches='tight'); plt.show()


## 7. Save artifacts and decision evidence

In [ ]:
all_path=PROCESSED_DIR/'fisheries_harmonized_all.parquet'; model_path=PROCESSED_DIR/'fisheries_harmonized_model_candidates.parquet'
all_records.to_parquet(all_path,index=False); model_candidates.to_parquet(model_path,index=False)
summary.to_csv(EVIDENCE_DIR/'harmonization_summary.csv',index=False); source_summary.to_csv(EVIDENCE_DIR/'source_mapping_summary.csv',index=False)
pd.DataFrame(MAPPINGS).to_csv(EVIDENCE_DIR/'semantic_mapping.csv',index=False)
manifest={'created_utc':pd.Timestamp.now(tz='UTC').isoformat(),'runtime':'Google Colab' if IN_COLAB else 'Local Windows',
 'assumptions':['weight fields treated as kilograms','rows treated as species catch lines','year-specific fishing dates preferred',
 '2017 coordinate labels reversed','2020 coordinate labels swapped','2022 issue sheet excluded from model candidates',
 'exact canonical duplicates excluded from model candidates','catch weight used instead of CPUE'],
 'all_records':len(all_records),'model_candidates':len(model_candidates),
 'outputs':[str(all_path.name),str(model_path.name)]}
with (EVIDENCE_DIR/'harmonization_manifest.json').open('w',encoding='utf-8') as f: json.dump(manifest,f,indent=2)
print('Saved processed artifacts:',all_path,model_path,sep='\n')


## 8. Next gate

Review retention, validity, date coverage, and the spatial plot. `03_data_integration.ipynb` may then restrict modelling candidates to the environmental coverage and align daily chlorophyll-a/SST values. The integration notebook must retain the harmonization flags and report how many records are excluded by environmental coverage.